<a href="https://colab.research.google.com/github/I-say/nvidia-dli-regularizacion/blob/main/notebooks/modulo-03-clasificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
#  Módulo 03 · Tu primer clasificador
### Curso de Regularización · Preparación NVIDIA DLI · CUGDL 2026
**Instructor:** Miguel Isay Morales Cortés · [@miguel_isay](https://instagram.com/miguel_isay)

---

## ¿Qué vamos a hacer?

Vamos a construir **un clasificador de frutas** desde cero — solo con Python.  
Sin IA, sin librerías de Deep Learning. Solo lógica.

¿Por qué? Porque el proyecto final del NVIDIA DLI hace exactamente esto:  
toma una imagen de fruta y decide si está **fresca** o **podrida**.

Si construyes la versión simple hoy, el día del workshop entenderás **qué está haciendo el modelo**  
aunque no entiendas cada línea de código de Keras.

> 💡 **La idea central:** Un clasificador es solo una función que recibe datos y devuelve una etiqueta.

---

## El problema

Tenemos frutas con estas características:
- `color` → qué tan oscura es la fruta (0 = muy oscura, 10 = muy brillante)
- `firmeza` → qué tan firme está (0 = muy blanda, 10 = muy dura)
- `dias` → cuántos días lleva en el almacén

Queremos decidir si está **fresca** o **podrida**.

Así se ve el "dataset" que usaremos:

In [ ]:
import numpy as np

# Nuestro dataset: cada fruta tiene [color, firmeza, dias]
# y una etiqueta: 1 = fresca, 0 = podrida

datos = [
    # [color, firmeza, dias], etiqueta
    ([8, 9, 1],  "fresca"),
    ([7, 8, 2],  "fresca"),
    ([9, 7, 1],  "fresca"),
    ([6, 6, 3],  "fresca"),
    ([3, 2, 8],  "podrida"),
    ([2, 1, 10], "podrida"),
    ([4, 3, 7],  "podrida"),
    ([1, 2, 9],  "podrida"),
]

print("Dataset de frutas:")
print(f"{'Color':>6} {'Firmeza':>8} {'Días':>6} {'Estado':>10}")
print("-" * 35)
for caracteristicas, etiqueta in datos:
    print(f"{caracteristicas[0]:>6} {caracteristicas[1]:>8} {caracteristicas[2]:>6} {etiqueta:>10}")

---
## Versión 1 — Clasificador por reglas simples

La forma más básica: definir reglas manualmente.

In [ ]:
def clasificador_v1(color, firmeza, dias):
    """
    Clasificador simple basado en reglas.
    Devuelve 'fresca' o 'podrida'.
    """
    if dias > 5:
        return "podrida"
    elif firmeza < 4:
        return "podrida"
    else:
        return "fresca"

# Probamos con las 8 frutas
print("Resultados del Clasificador V1:")
print(f"{'Real':>10} {'Predicción':>12} {'Correcto?':>10}")
print("-" * 36)

correctos = 0
for caracteristicas, etiqueta_real in datos:
    prediccion = clasificador_v1(caracteristicas[0], caracteristicas[1], caracteristicas[2])
    correcto = "✓" if prediccion == etiqueta_real else "✗"
    if prediccion == etiqueta_real:
        correctos += 1
    print(f"{etiqueta_real:>10} {prediccion:>12} {correcto:>10}")

precision = correctos / len(datos)
print(f"\nPrecisión: {correctos}/{len(datos)} = {precision:.0%}")

---
## Versión 2 — Clasificador con puntuación (más parecido a la IA)

En vez de reglas fijas, calculamos una **puntuación de frescura**.  
Si la puntuación es alta → fresca. Si es baja → podrida.

Esto es exactamente la lógica detrás de las redes neuronales:  
calculan un número (probabilidad) y si supera un umbral, predicen una clase.

In [ ]:
def calcular_puntaje_frescura(color, firmeza, dias):
    """
    Calcula un puntaje de frescura entre 0 y 1.
    
    - color y firmeza altos → más fresca
    - dias altos → menos fresca
    """
    # Normalizamos cada variable a escala 0-1
    color_norm    = color / 10.0          # 0-10 → 0.0-1.0
    firmeza_norm  = firmeza / 10.0        # 0-10 → 0.0-1.0
    dias_norm     = 1 - (dias / 14.0)    # 0-14 días, invertido: más días = menos fresco
    
    # Ponderamos: firmeza es lo más importante
    puntaje = (color_norm * 0.2) + (firmeza_norm * 0.5) + (dias_norm * 0.3)
    return round(puntaje, 3)

def clasificador_v2(color, firmeza, dias, umbral=0.5):
    """
    Clasificador basado en puntaje.
    Si el puntaje >= umbral → fresca, si no → podrida.
    """
    puntaje = calcular_puntaje_frescura(color, firmeza, dias)
    if puntaje >= umbral:
        return "fresca", puntaje
    else:
        return "podrida", puntaje

# Probamos con las 8 frutas
print("Resultados del Clasificador V2 (con puntaje):")
print(f"{'Real':>10} {'Predicción':>12} {'Puntaje':>9} {'Correcto?':>10}")
print("-" * 46)

correctos = 0
for caract, etiqueta_real in datos:
    prediccion, puntaje = clasificador_v2(caract[0], caract[1], caract[2])
    correcto = "✓" if prediccion == etiqueta_real else "✗"
    if prediccion == etiqueta_real:
        correctos += 1
    print(f"{etiqueta_real:>10} {prediccion:>12} {puntaje:>9} {correcto:>10}")

precision = correctos / len(datos)
print(f"\nPrecisión: {correctos}/{len(datos)} = {precision:.0%}")

---
## La conexión con el NVIDIA DLI

Lo que acabas de construir es la **misma lógica** que usa el proyecto final del workshop,  
pero en vez de reglas manuales, **la red neuronal aprende los pesos sola**.

Compara:

In [ ]:
# LO QUE HICIMOS NOSOTROS:
# puntaje = (color * 0.2) + (firmeza * 0.5) + (dias * 0.3)

# LO QUE HACE UNA RED NEURONAL (pseudocódigo simplificado):
# puntaje = (pixel_1 * w1) + (pixel_2 * w2) + ... + (pixel_N * wN) + bias
#
# La diferencia: los pesos (w1, w2...) no los defines tú.
# El modelo los APRENDE automáticamente durante el entrenamiento.

print("Nuestro clasificador:")
print("  puntaje = (color × 0.2) + (firmeza × 0.5) + (días × 0.3)")
print("  → los pesos los pusimos nosotros")
print()
print("Red neuronal del NVIDIA DLI:")
print("  predicción = f(píxeles × pesos_aprendidos + bias)")
print("  → los pesos los aprende el modelo solo")
print()
print("La lógica es EXACTAMENTE la misma.")
print("La diferencia es que la red puede aprender millones de pesos")
print("a partir de miles de ejemplos, mucho mejor de lo que nosotros podríamos.")

---
##  Ejercicio — Mejora el clasificador

El clasificador V2 usa estos pesos: color=0.2, firmeza=0.5, días=0.3  

Tu tarea: experimenta cambiando los pesos y el umbral para ver si puedes mejorar la precisión.  
Pista: intenta hacer que todos los pesos sumen 1.0

Pregunta de reflexión: ¿Qué le estás haciendo manualmente cuando cambias los pesos?  
*(Respuesta: lo mismo que hace el entrenamiento de una red neuronal, pero a mano)*

In [ ]:
# Modifica los pesos aquí y observa cómo cambia la precisión
PESO_COLOR   = 0.2   # ← cambia este valor
PESO_FIRMEZA = 0.5   # ← y este
PESO_DIAS    = 0.3   # ← y este
UMBRAL       = 0.5   # ← y este

def clasificador_ajustable(color, firmeza, dias):
    color_norm   = color / 10.0
    firmeza_norm = firmeza / 10.0
    dias_norm    = 1 - (dias / 14.0)
    
    puntaje = (color_norm * PESO_COLOR) + (firmeza_norm * PESO_FIRMEZA) + (dias_norm * PESO_DIAS)
    
    if puntaje >= UMBRAL:
        return "fresca", round(puntaje, 3)
    else:
        return "podrida", round(puntaje, 3)

# Evaluación
correctos = 0
for caract, etiqueta_real in datos:
    prediccion, puntaje = clasificador_ajustable(caract[0], caract[1], caract[2])
    if prediccion == etiqueta_real:
        correctos += 1

print(f"Pesos: color={PESO_COLOR}, firmeza={PESO_FIRMEZA}, días={PESO_DIAS}, umbral={UMBRAL}")
print(f"Precisión: {correctos}/{len(datos)} = {correctos/len(datos):.0%}")

---
##  Resumen del módulo

| Concepto | Conexión con NVIDIA DLI |
|----------|--------------------------|
| Clasificador | El proyecto final del workshop clasifica frutas frescas/podridas |
| Puntaje/probabilidad | Las redes neuronales devuelven un número entre 0 y 1 |
| Umbral (0.5) | Keras usa `sigmoid` que convierte a probabilidad, misma idea |
| Pesos manuales | El entrenamiento aprende los pesos automáticamente |
| Precisión | La métrica `accuracy` del workshop mide lo mismo que calculamos aquí |

---
** Siguiente módulo:** [Módulo 04 · Visualización básica — ver datos como los ve la IA](./modulo-04-visualizacion.ipynb)

---
*Curso de Regularización NVIDIA DLI · CUGDL 2026 · Instructor: Miguel Isay Morales Cortés*